# Exploratory Data Analysis (EDA) - Image Classification Dataset

Ce notebook explore le dataset d'images pour comprendre sa structure, visualiser des exemples et analyser les statistiques.


In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline


## 1. Chargement du Dataset

Nous allons utiliser CIFAR-10 pour la démonstration. Vous pouvez également utiliser un dataset local avec la structure `data/train/<class>/*` et `data/val/<class>/*`.


In [ ]:
# Option 1: Utiliser CIFAR-10 (démo rapide)
USE_CIFAR10 = True  # Mettre à False pour utiliser un dataset local

if USE_CIFAR10:
    from src.dataset import load_cifar10_dataset
    
    train_loader, val_loader, class_names = load_cifar10_dataset(
        data_dir='../data',
        img_size=224,
        batch_size=32,
        num_workers=2
    )
    print(f"Dataset: CIFAR-10")
    print(f"Nombre de classes: {len(class_names)}")
    print(f"Classes: {class_names}")
else:
    # Option 2: Utiliser un dataset local
    from src.dataset import load_imagefolder_dataset
    
    train_loader, val_loader, class_names = load_imagefolder_dataset(
        data_dir='../data',
        img_size=224,
        batch_size=32,
        num_workers=2
    )
    print(f"Dataset: ImageFolder (local)")
    print(f"Nombre de classes: {len(class_names)}")
    print(f"Classes: {class_names}")


## 2. Statistiques du Dataset


In [ ]:
# Compter les échantillons par classe dans le train set
train_labels = []
for _, labels in train_loader:
    train_labels.extend(labels.numpy())

train_class_counts = Counter(train_labels)
print("Distribution des classes (Train):")
for class_idx, count in sorted(train_class_counts.items()):
    print(f"  {class_names[class_idx]}: {count} échantillons")

# Compter les échantillons par classe dans le validation set
val_labels = []
for _, labels in val_loader:
    val_labels.extend(labels.numpy())

val_class_counts = Counter(val_labels)
print("\nDistribution des classes (Validation):")
for class_idx, count in sorted(val_class_counts.items()):
    print(f"  {class_names[class_idx]}: {count} échantillons")


In [ ]:
# Visualiser la distribution des classes
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Train set
train_counts = [train_class_counts[i] for i in range(len(class_names))]
axes[0].bar(class_names, train_counts, color='steelblue', alpha=0.7)
axes[0].set_title('Distribution des Classes - Train Set', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classe', fontsize=12)
axes[0].set_ylabel('Nombre d\'échantillons', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Validation set
val_counts = [val_class_counts[i] for i in range(len(class_names))]
axes[1].bar(class_names, val_counts, color='coral', alpha=0.7)
axes[1].set_title('Distribution des Classes - Validation Set', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Classe', fontsize=12)
axes[1].set_ylabel('Nombre d\'échantillons', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTotal train: {sum(train_counts)} échantillons")
print(f"Total validation: {sum(val_counts)} échantillons")


## 3. Visualisation d'Exemples d'Images


In [ ]:
# Fonction pour dénormaliser une image
def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """Dénormalise un tenseur d'image"""
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor

# Récupérer un batch d'images
data_iter = iter(train_loader)
images, labels = next(data_iter)

# Visualiser quelques images
num_images = 16
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.ravel()

for idx in range(num_images):
    img = images[idx]
    label = labels[idx].item()
    
    # Dénormaliser
    img = denormalize(img.clone())
    img = torch.clamp(img, 0, 1)
    
    # Convertir en numpy et afficher
    img_np = img.permute(1, 2, 0).numpy()
    axes[idx].imshow(img_np)
    axes[idx].set_title(f'{class_names[label]}', fontsize=10, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Exemples d\'Images du Dataset (Train)', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


## 4. Visualisation par Classe


In [ ]:
# Afficher quelques exemples pour chaque classe
num_examples_per_class = 5
fig, axes = plt.subplots(len(class_names), num_examples_per_class, figsize=(15, 3*len(class_names)))

# Collecter des exemples pour chaque classe
class_examples = {i: [] for i in range(len(class_names))}
for images, labels in train_loader:
    for img, label in zip(images, labels):
        if len(class_examples[label.item()]) < num_examples_per_class:
            class_examples[label.item()].append((img, label.item()))
    
    # Vérifier si on a assez d'exemples
    if all(len(examples) >= num_examples_per_class for examples in class_examples.values()):
        break

# Afficher
for class_idx, class_name in enumerate(class_names):
    for example_idx, (img, label) in enumerate(class_examples[class_idx]):
        ax = axes[class_idx, example_idx]
        
        # Dénormaliser
        img_denorm = denormalize(img.clone())
        img_denorm = torch.clamp(img_denorm, 0, 1)
        img_np = img_denorm.permute(1, 2, 0).numpy()
        
        ax.imshow(img_np)
        if example_idx == 0:
            ax.set_ylabel(class_name, fontsize=12, fontweight='bold', rotation=0, ha='right', va='center')
        ax.axis('off')

plt.suptitle('Exemples par Classe', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Analyse des Dimensions et Statistiques des Images


In [ ]:
# Analyser les statistiques des images
sample_images = []
for images, _ in train_loader:
    sample_images.append(images)
    if len(sample_images) >= 10:  # Prendre 10 batches
        break

all_images = torch.cat(sample_images, dim=0)

# Statistiques
print("Statistiques des Images:")
print(f"  Shape: {all_images.shape}")
print(f"  Min value: {all_images.min().item():.4f}")
print(f"  Max value: {all_images.max().item():.4f}")
print(f"  Mean: {all_images.mean().item():.4f}")
print(f"  Std: {all_images.std().item():.4f}")

# Histogramme des valeurs de pixels
plt.figure(figsize=(10, 6))
plt.hist(all_images.numpy().flatten(), bins=100, alpha=0.7, color='steelblue', edgecolor='black')
plt.title('Distribution des Valeurs de Pixels', fontsize=14, fontweight='bold')
plt.xlabel('Valeur de Pixel', fontsize=12)
plt.ylabel('Fréquence', fontsize=12)
plt.grid(alpha=0.3)
plt.show()


## 6. Résumé

Ce notebook a exploré le dataset d'images. Les principales observations sont:
- Nombre de classes et leur distribution
- Exemples visuels pour chaque classe
- Statistiques des images

Le prochain notebook (`02_training_transfer_learning.ipynb`) utilisera ces données pour entraîner un modèle avec transfer learning.
